# 1. `CIFParser`


## 1.1 Loading from `mmCIF`

In [10]:
from cifutils import CIFParser
from pathlib import Path

# ...instantiate the parser
parser = CIFParser()

# ...define the path to the CIF file
# (.cif, .bcif, .cif.gz, and .bcif.gz files are supported)
path = '/databases/rcsb/cif/ne/3ne2.cif.gz'

# ...parse the CIF file, enumerating non-default parameters
# (See `parser.parse` documentation within `cifutils` for more details)
result_dict = parser.parse(
    filename=Path(path),
    build_assembly="1",
    add_bonds=True,
    add_missing_atoms=True,
    remove_waters=True,
    patch_symmetry_centers=True,
    convert_mse_to_met=True,
    fix_arginines=True,
    keep_hydrogens=False,
    model=1,
)

# ...print the keys of the resulting dictionary
print("Keys:")
print(list(result_dict.keys()))


Keys:
['chain_info', 'ligand_info', 'atom_array_stack', 'assemblies', 'metadata', 'extra_info']


In [15]:
from cifutils.utils.visualize import view
atom_array = result_dict['assemblies']["1"][0]
atom_array = atom_array[atom_array.occupancy > 0]

# ...visualize the assymmetric unit
view(atom_array)

Chain A - Protein: True, Nucleic: False, Ion: False
Chain B - Protein: True, Nucleic: False, Ion: False
Chain C - Protein: True, Nucleic: False, Ion: False
Chain D - Protein: True, Nucleic: False, Ion: False
Chain I - Protein: False, Nucleic: False, Ion: False
Chain J - Protein: False, Nucleic: False, Ion: False
Chain K - Protein: False, Nucleic: False, Ion: False
Chain L - Protein: False, Nucleic: False, Ion: False
Chain M - Protein: False, Nucleic: False, Ion: False


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [19]:
# ...print the chain information
# (A list of dictionaries, each containing information about a chain)
result_dict["chain_info"]

{'A': {'rcsb_entity': '1',
  'type': 'polypeptide(L)',
  'unprocessed_entity_canonical_sequence': 'MTMTLAKRFTAEVVGTFILVFFGPGAAVITLMIANGADKPNEFNIGIGALGGLGDWFAIGMAFALAIAAVIYSLGRISGAHINPAVTIALWSIGRFPGREVVPYIVAQFIGAALGSLLFLACVGPAAATVGGLGATAPFPGIGYGQAILTEAIGTFLLMLVIMGVAVDERAPPGFAGLVIGLTVGGIITTIGNITGSSLNPARTFGPYLGDSLMGINLWQYFPIYVIGPIVGAVAAAWLYNYLAKE',
  'unprocessed_entity_non_canonical_sequence': 'MTMTLAKRFTAEVVGTFILVFFGPGAAVITLMIANGADKPNEFNIGIGALGGLGDWFAIGMAFALAIAAVIYSLGRISGAHINPAVTIALWSIGRFPGREVVPYIVAQFIGAALGSLLFLACVGPAAATVGGLGATAPFPGIGYGQAILTEAIGTFLLMLVIMGVAVDERAPPGFAGLVIGLTVGGIITTIGNITGSSLNPARTFGPYLGDSLMGINLWQYFPIYVIGPIVGAVAAAWLYNYLAKE',
  'is_polymer': True,
  'ec_numbers': [],
  'residue_name_list': ['MET',
   'THR',
   'MET',
   'THR',
   'LEU',
   'ALA',
   'LYS',
   'ARG',
   'PHE',
   'THR',
   'ALA',
   'GLU',
   'VAL',
   'VAL',
   'GLY',
   'THR',
   'PHE',
   'ILE',
   'LEU',
   'VAL',
   'PHE',
   'PHE',
   'GLY',
   'PRO',
   'GLY',
   'ALA',
   'ALA',
   'VAL',
   'ILE',
  

In [20]:
# ...print the ligand information
# (A list of dictionaries, each containing information about a ligand. For example, the ligand of interest, or the ligand fit-to-density)
result_dict["ligand_info"]

{'ligand_of_interest': [], 'has_ligand_of_interest': False}

In [21]:
# ...print the metadata
result_dict["metadata"]

{'id': '3ne2',
 'method': 'X-RAY_DIFFRACTION',
 'deposition_date': '2010-06-08',
 'release_date': '2010-09-22',
 'resolution': 3.0,
 'extra_metadata': None}

## 1.2 Manipulating the `AtomArray`

In [13]:
# ...extract the first assembly, first model
atom_array = result_dict["assemblies"]["1"][0]

# ...get the coordinates of all of the resolved CA atoms
print("Coordinates of all CA atoms:")
ca = atom_array[(atom_array.atom_name == "CA") & (atom_array.occupancy >0)]
print(ca.coord.shape)

# ...get the coordinates of chain A
print("Coordinates of chain A (all heavy atoms):")
chain = atom_array[atom_array.chain_id == "A"]
print(chain.coord.shape)

Coordinates of all CA atoms:
(978, 3)
Coordinates of chain A (all heavy atoms):
(1782, 3)


In [42]:
# ...check bonds
atom_array.bonds.as_array()

array([[   2,    7,    1],
       [   9,   15,    1],
       [  17,   22,    1],
       ...,
       [7129, 7130,    1],
       [7130, 7131,    1],
       [7131, 7132,    1]], dtype=uint32)

## 1.3 Basic distance computations

In [14]:
import biotite.structure as struc
import numpy as np

### ATOM TO ATOM
# Distance between two atoms
distance = struc.distance(ca.coord[0], ca.coord[1])  # Distance between first two C-alpha atoms
print(f"Distance between first two C-alpha atoms: {distance:.2f} Å")

Distance between first two C-alpha atoms: 3.87 Å


In [15]:
### ARRAY TO ATOM
# Distance between one atom and all other atoms in the array
distance = struc.distance(ca[0], ca)
print(f"Distances between first C-alpha atom and all other C-alpha atoms: {distance}")

Distances between first C-alpha atom and all other C-alpha atoms: [ 0.         3.867667   6.6482844  8.908499  11.5260725 10.039213
  9.38693   12.926718  14.379794  12.894565  14.713824  17.743765
 18.141035  17.816875  20.372517  22.463972  22.963528  23.552736
 26.406012  27.727478  28.550404  28.864363  31.484188  32.607037
 34.271465  36.060665  37.471706  38.642437  40.23522   42.18375
 43.12232   44.50063   46.16822   47.6207    51.363705  51.566418
 53.569695  51.51129   53.774006  52.88923   51.13428   48.153595
 47.39715   46.006397  45.704758  42.715645  44.591293  47.65415
 47.54075   45.16692   45.4869    42.541542  41.314404  40.40863
 38.13826   36.512444  36.13391   34.313705  32.20724   31.219013
 30.395615  27.919043  26.757065  26.177673  24.316687  22.05974
 21.494232  20.855267  18.10323   16.246792  16.694527  15.451185
 12.777143  11.915175   9.117025   7.704437   9.109868  10.59693
 13.7125435 15.700364  18.12988   20.399778  20.881332  22.801842
 19.808987  17.

## 1.4 `CellList` - Efficient distance computations

In [17]:
# ...filter to resolved atoms
resolved_atom_array = atom_array[atom_array.occupancy > 0]

# ...build CellList
cell_list = struc.CellList(resolved_atom_array, cell_size=5.0)

In [18]:
### ATOMS NEAR ATOM
# Get all atoms within 5 Å of the first atom
print(f"Target atom: {resolved_atom_array[0].coord} {resolved_atom_array[0].atom_name} {resolved_atom_array[0].res_name} {resolved_atom_array[0].res_id}")
near_atoms = cell_list.get_atoms(resolved_atom_array[0].coord, radius=4)
print(f"Number of atoms within 7 Å of the first atom: {near_atoms.shape[0]}")
print(f"Atom indices: {near_atoms}")
chain_ids = resolved_atom_array.chain_id[near_atoms]
print(f"Chain IDs: {chain_ids}")
residue_ids = resolved_atom_array.res_id[near_atoms]
print(f"Residue IDs: {residue_ids}")
print(f"Residue names: {resolved_atom_array.res_name[near_atoms]}")

Target atom: [-20.452  14.325  -0.824] N THR 2
Number of atoms within 7 Å of the first atom: 8
Atom indices: [0 1 3 2 7 4 5 6]
Chain IDs: ['A' 'A' 'A' 'A' 'A' 'A' 'A' 'A']
Residue IDs: [2 2 2 2 3 2 2 2]
Residue names: ['THR' 'THR' 'THR' 'THR' 'MET' 'THR' 'THR' 'THR']


# 2. `Datahub`

## 2.1 Pre-built PDB datasets on disk

In [3]:
import pandas as pd

# ...load the dataframes from the parquet files
PN_UNITS_DF = pd.read_parquet("/home/ncorley/projects/datahub/tests/data/pn_units_df.parquet")
INTERFACES_DF = pd.read_parquet("/home/ncorley/projects/datahub/tests/data/interfaces_df.parquet")

# ...as an example, print the first few rows of the PN_UNITS_DF dataframe
PN_UNITS_DF


,pdb_id,assembly_id,clash_severity,resolution,deposition_date,release_date,method,num_polymer_pn_units,q_pn_unit_iid,q_pn_unit_id,...,all_pn_unit_iids,n_prot,n_nuc,n_ligand,example_id,"q_pn_unit_protein_cluster_(id:0,4)(cov:0,8)(cov_mode:0)_rep_seq_hash","q_pn_unit_nucleic_acid_cluster_(id:0,4)(cov:0,8)(cov_mode:0)_rep_seq_hash","q_pn_unit_protein_cluster_(id:1,0)(cov:0,8)(cov_mode:0)_rep_seq_hash","q_pn_unit_nucleic_acid_cluster_(id:1,0)(cov:0,8)(cov_mode:0)_rep_seq_hash",cluster
253,3gfi,1,ClashSeverity.NO_CLASH,2.10,2009-02-26,2009-08-25,X-RAY_DIFFRACTION,4,C_1,C,...,"[""C_1"", ""B_1"", ""D_1"", ""A_1""]",0,1,0,"{['pdb', 'pn_units']}{3gfi}{1}{['C_1']}",None,b1cf78ebf85,None,b1cf78ebf85,b1cf78ebf85
254,3gfi,1,ClashSeverity.NO_CLASH,2.10,2009-02-26,2009-08-25,X-RAY_DIFFRACTION,4,B_1,B,...,"[""C_1"", ""B_1"", ""D_1"", ""A_1""]",1,0,0,"{['pdb', 'pn_units']}{3gfi}{1}{['B_1']}",73a6ef54b61,None,e881ff318a2,None,73a6ef54b61
255,3gfi,1,ClashSeverity.NO_CLASH,2.10,2009-02-26,2009-08-25,X-RAY_DIFFRACTION,4,D_1,D,...,"[""C_1"", ""B_1"", ""D_1"", ""A_1""]",0,1,0,"{['pdb', 'pn_units']}{3gfi}{1}{['D_1']}",None,084b35938fe,None,084b35938fe,084b35938fe
256,3gfi,1,ClashSeverity.NO_CLASH,2.10,2009-02-26,2009-08-25,X-RAY_DIFFRACTION,4,A_1,A,...,"[""C_1"", ""B_1"", ""D_1"", ""A_1""]",1,0,0,"{['pdb', 'pn_units']}{3gfi}{1}{['A_1']}",73a6ef54b61,None,e881ff318a2,None,73a6ef54b61
588,4cip,1,ClashSeverity.NO_CLASH,1.22,2013-12-13,2014-05-21,X-RAY_DIFFRACTION,2,C_1,C,...,"[""C_1"", ""A_2"", ""B_2"", ""C_2"", ""B_1"", ""A_1""]",0,0,1,"{['pdb', 'pn_units']}{4cip}{1}{['C_1']}",None,None,None,None,ASC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3360661,1z8r,1,ClashSeverity.NO_CLASH,NaN,2005-03-31,2006-02-14,SOLUTION_NMR,1,B_1,B,...,"[""B_1"", ""A_1""]",0,0,1,"{['pdb', 'pn_units']}{1z8r}{1}{['B_1']}",None,None,None,None,ZN
3360662,1z8r,1,ClashSeverity.NO_CLASH,NaN,2005-03-31,2006-02-14,SOLUTION_NMR,1,A_1,A,...,"[""B_1"", ""A_1""]",1,0,0,"{['pdb', 'pn_units']}{1z8r}{1}{['A_1']}",e8a549ed2e0,None,53a112a9f13,None,e8a549ed2e0
3362366,3nr5,1,ClashSeverity.NO_CLASH,1.55,2010-06-30,2010-10-13,X-RAY_DIFFRACTION,1,A_1,A,...,"[""A_1""]",1,0,0,"{['pdb', 'pn_units']}{3nr5}{1}{['A_1']}",5ea518f0dc9,None,5ea518f0dc9,None,5ea518f0dc9
3364553,1vdw,1,ClashSeverity.NO_CLASH,1.30,2004-03-25,2004-04-06,X-RAY_DIFFRACTION,2,B_1,B,...,"[""B_1"", ""A_1""]",1,0,0,"{['pdb', 'pn_units']}{1vdw}{1}{['B_1']}",2bf532d0756,None,f1a4d1c4cc6,None,2bf532d0756


In [47]:
print(PN_UNITS_DF.columns)

Index(['pdb_id', 'assembly_id', 'clash_severity', 'resolution',
       'deposition_date', 'release_date', 'method', 'num_polymer_pn_units',
       'q_pn_unit_iid', 'q_pn_unit_id', 'q_pn_unit_type',
       'q_pn_unit_transformation_id', 'q_pn_unit_num_atoms',
       'q_pn_unit_is_multichain', 'q_pn_unit_is_multiresidue',
       'q_pn_unit_num_resolved_residues', 'q_pn_unit_is_metal',
       'q_pn_unit_is_loi', 'q_pn_unit_ligand_validity',
       'q_pn_unit_bonded_polymer_pn_units', 'q_pn_unit_non_polymer_res_names',
       'q_pn_unit_ec_numbers', 'q_pn_unit_sequence_length',
       'q_pn_unit_processed_entity_canonical_sequence',
       'q_pn_unit_processed_entity_non_canonical_sequence',
       'q_pn_unit_processed_entity_canonical_sequence_hash',
       'q_pn_unit_processed_entity_non_canonical_sequence_hash',
       'q_pn_unit_primary_polymer_partner',
       'q_pn_unit_contacting_pn_unit_iids', 'q_pn_unit_close_pn_unit_iids',
       'all_pn_unit_iids', 'n_prot', 'n_nuc', 'n_ligand',

In [52]:
from cifutils.enums import ChainType

# ...get all RNA examples in the PDB
PN_UNITS_DF_RNA = PN_UNITS_DF[PN_UNITS_DF.q_pn_unit_type== ChainType.RNA.value]
print("RNA Dataframe:", PN_UNITS_DF_RNA.shape)

# ...get all examples with covalent modifications
PN_UNITS_DF_COVALENT_MODIFICATIONS = PN_UNITS_DF[PN_UNITS_DF.q_pn_unit_bonded_polymer_pn_units != "set()"]
print("Covalent Modifications Dataframe:", PN_UNITS_DF_COVALENT_MODIFICATIONS.shape)

RNA Dataframe: (408, 40)
Covalent Modifications Dataframe: (2377, 40)


In [11]:
from datahub.datasets.dataframe_parsers import InterfacesDFParser, PNUnitsDFParser
from datahub.datasets.pdb_dataset import PDBDataset
from datahub.pipelines.rf2aa import build_rf2aa_transform_pipeline
from datahub.datasets.base import NamedConcatDataset
from pathlib import Path

# ...define the paths to the MSA directories
PROTEIN_MSA_DIR = Path("/projects/ml/RF2_allatom/data_preprocessing/msa/protein")
RNA_MSA_DIR = Path("/projects/ml/RF2_allatom/data_preprocessing/msa/rna")

# ...define the filters to apply to the dataframes
SHARED_TEST_FILTERS = [
    "deposition_date < '2022-01-01'",
    "resolution < 5.0 and ~method.str.contains('NMR')",
    "num_polymer_pn_units <= 20",  # To ensure we don't freeze loading a massive example
    "cluster.notnull()",
    "method in ['X-RAY_DIFFRACTION', 'ELECTRON_MICROSCOPY']",
]

# Define the PDB datasets with their respective parsers...
RF2AA_PN_UNITS_DATASET = PDBDataset(
    name="pn_units",
    dataset_path=PN_UNITS_DF,
    cif_parser=parser,
    filters=SHARED_TEST_FILTERS,
    dataset_parser=PNUnitsDFParser(), # This is the parser for the PNUnitsDF dataframe; it will be used to extract the data from the dataframe (e.g., which example to load, contacting molecules, sequence information, etc.)
    id_column="example_id",
    transform=build_rf2aa_transform_pipeline(
        protein_msa_dir=PROTEIN_MSA_DIR,
        rna_msa_dir=RNA_MSA_DIR,
        n_recycles=5,
        crop_size=256,
        crop_contiguous_probability=1 / 3,
        crop_spatial_probability=2 / 3,
    ),
    unpack_data_dict=False,
)

RF2AA_INTERFACES_DATASET = PDBDataset(
    name="interfaces",
    dataset_path=INTERFACES_DF,
    cif_parser=parser,
    filters=SHARED_TEST_FILTERS,
    dataset_parser=InterfacesDFParser(),
    id_column="example_id",
    transform=build_rf2aa_transform_pipeline(
        protein_msa_dir=PROTEIN_MSA_DIR,
        rna_msa_dir=RNA_MSA_DIR,
        n_recycles=5,
        crop_size=256,
        crop_spatial_probability=1.0,
        crop_contiguous_probability=0.0,
    ),
    unpack_data_dict=False,
)

# ...build the ConcatDataset
RF2AA_PDB_DATASET = NamedConcatDataset(
    datasets=[RF2AA_PN_UNITS_DATASET, RF2AA_INTERFACES_DATASET], name="pdb"
) 

## 2.2 Samplers

## 2.3 Transforms

In [13]:
from datahub.transforms.atom_array import (
    RemoveHydrogens,
    RemoveTerminalOxygen,
    AddGlobalTokenIdAnnotation,
    AddGlobalAtomIdAnnotation,
    SortPolyThenNonPoly
)
from datahub.transforms.atomize import AtomizeResidues
from datahub.transforms.crop import CropSpatialLikeAF3
from datahub.transforms.encoding import EncodeAtomArray
from datahub.encoding_definitions import RF2_ATOM36_ENCODING
from datahub.transforms.openbabel_utils import AddOpenBabelMoleculesForAtomizedMolecules
from datahub.transforms.base import Compose
from datahub.datasets.dataframe_parsers import load_from_row
from datahub.utils.rng import create_rng_state_from_seeds, rng_state

seed = 3
encoding = RF2_ATOM36_ENCODING
pdb_id = "3ne2"

row = PN_UNITS_DF[PN_UNITS_DF["pdb_id"] == pdb_id].iloc[0]  # Get the first row; we don't care which we choose
data = load_from_row(row, PNUnitsDFParser(), cif_parser=parser)

print("Keys before:", data.keys())

pipe = Compose(
    [
        RemoveHydrogens(),
        AddGlobalAtomIdAnnotation(),
        RemoveTerminalOxygen(),
        AtomizeResidues(atomize_by_default=True, res_names_to_ignore=encoding.tokens, move_atomized_part_to_end=True),
        AddGlobalTokenIdAnnotation(),
        SortPolyThenNonPoly(),
        AddOpenBabelMoleculesForAtomizedMolecules(),
        CropSpatialLikeAF3(crop_size=50, keep_uncropped_atom_array=True),
        EncodeAtomArray(encoding=encoding),
    ],
    track_rng_state=False,
)

with rng_state(create_rng_state_from_seeds(np_seed=seed, torch_seed=seed, py_seed=seed)):
    data = pipe(data)
    print("Keys after:", data.keys())

Keys before: dict_keys(['example_id', 'pdb_id', 'assembly_id', 'query_pn_unit_iids', 'extra_info', 'atom_array', 'atom_array_stack', 'chain_info', 'ligand_info', 'metadata'])
Keys after: dict_keys(['example_id', 'pdb_id', 'assembly_id', 'query_pn_unit_iids', 'extra_info', 'atom_array', 'atom_array_stack', 'chain_info', 'ligand_info', 'metadata', 'openbabel', 'crop_info', 'encoded'])


In [16]:
view(data["atom_array"])

Chain C - Protein: True, Nucleic: False, Ion: False
Chain D - Protein: True, Nucleic: False, Ion: False


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
from datahub.transforms.msa.msa import (
    EncodeMSA,
    FeaturizeMSALikeAF3,
    FeaturizeMSALikeRF2AA,
    FillFullMSAFromEncoded,
    LoadPolymerMSAs,
    PairAndMergePolymerMSAs,
    get_full_msa_profile_and_insertion_mean,
)

    row = PN_UNITS_DF[PN_UNITS_DF["pdb_id"] == pdb_id].iloc[0]  # Get the first row; we don't care which we choose
    data = load_from_row(row, PNUnitsDFParser(), cif_parser=CIF_PARSER)

    encoding = RF2AA_ATOM36_ENCODING
    res_names_to_ignore = encoding.tokens[
        encoding.tokens != "ASP"
    ]  # Atomize aspartate so we can test atomization and MSA indexing
    PAD_TOKEN = RF2AA_ATOM36_ENCODING.token_to_idx["UNK"]

    # Apply initial transforms
    # fmt: off
    pipeline = Compose([
        AddWithinPolyResIdxAnnotation(),
        LoadPolymerMSAs(protein_msa_dir=PROTEIN_MSA_DIR, rna_msa_dir=RNA_MSA_DIR, max_msa_sequences=100),
        PairAndMergePolymerMSAs(),
        AtomizeResidues(
            atomize_by_default=True, res_names_to_ignore=res_names_to_ignore, move_atomized_part_to_end=False
        ),
        EncodeAtomArray(encoding),
        # MSA featurize workflow
        EncodeMSA(encoding=encoding, token_to_use_for_gap=PAD_TOKEN),
        FillFullMSAFromEncoded(pad_token=PAD_TOKEN),
    ], track_rng_state=False)
    # fmt: on

    output = pipeline(data)
    atom_array = output["atom_array"]